# Reporting Metrics Viewer

Created with help from Codex

This notebook shows the two evaluation metrics you need to report for both models:

- `routed_blend`
- `neural` (`mlp_stacking`)

Metrics shown:
- **AUC**
- **TPR@FPR = 0.1**

## 1. Load the metrics file

In [22]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, roc_curve
from IPython.display import display

METRICS_PATH = Path("metrics.json")
PRED_PATH = Path("val_predictions.csv")

## 2. Read `metrics.json`

In [23]:
if not METRICS_PATH.exists():
    raise FileNotFoundError(f"Could not find {METRICS_PATH.resolve()}")
if not PRED_PATH.exists():
    raise FileNotFoundError(f"Could not find {PRED_PATH.resolve()}")

with METRICS_PATH.open() as f:
    metrics = json.load(f)

pred_df = pd.read_csv(PRED_PATH)
pred_df.head()

,id,label,is_member
0,cauldron_ln_18258,0,0.001290
1,mix_SI_UT_4500_263,0,0.000430
2,cauldron_ln_27714,0,0.001626
3,mix_UI_ST_4500_189,0,0.000432
4,cauldron_ln_21048,1,0.001493


## 3. Compare `routed_blend` and `neural`

In [24]:
best_model = metrics.get("best_model", "unknown")

y_true = pred_df["label"]
y_score = pred_df["is_member"]

best_auc = float(roc_auc_score(y_true, y_score))
fpr, tpr, _ = roc_curve(y_true, y_score)
mask = fpr <= 0.1
if mask.any():
    best_tpr_at_fpr01 = float(tpr[mask].max())
else:
    best_tpr_at_fpr01 = float(tpr[np.argmin(np.abs(fpr - 0.1))])

routed_tpr = metrics.get("routed_blend_tpr_at_fpr01", np.nan)
neural_tpr = metrics.get("mlp_tpr_at_fpr01", np.nan)

if best_model == "routed_blend":
    routed_tpr = best_tpr_at_fpr01
elif best_model == "mlp_stacking":
    neural_tpr = best_tpr_at_fpr01

report_df = pd.DataFrame(
    {
        "model": ["routed_blend", "neural_mlp"],
        "AUC": [metrics.get("routed_blend_auc", np.nan), metrics.get("mlp_stacking_auc", np.nan)],
        "TPR@FPR=0.1": [routed_tpr, neural_tpr],
    }
).set_index("model").round(4)

print(f"Best model in this run: {best_model}")
display(report_df)

Best model in this run: routed_blend


,AUC,TPR@FPR=0.1
model,,
routed_blend,0.8444,0.2833
neural_mlp,0.8216,0.2167


## 4. Optional export

In [25]:
report_path = Path("report_metrics.csv")
report_df.to_csv(report_path, index=False)
print(f"Saved -> {report_path.resolve()}")

Saved -> /Users/jennifer/git/MDS-CL/COLX_531_TCOB/milestone6/src/NeuralModel_NeighborFeatures/report_metrics.csv
